# RQ1: yếu tố nào ảnh hưởng nhiều nhất tới năng lượng mỗi phiên sạc?

Notebook phân tích mức phiên sạc (một dòng là một phiên) trên toàn bộ ACN-Data đã làm sạch (ba site).
Biến mục tiêu là `kWhDelivered`. Các yếu tố cần xếp hạng: giờ trong ngày, loại địa điểm (khuôn viên công cộng so với nơi làm việc), ngày trong tuần.

Chia thời gian theo `connectionTime`: Train đến hết 31/10/2019, Val từ 01/11/2019 đến hết 15/01/2020, Test từ 16/01/2020 đến hết dữ liệu.
Hai lần huấn luyện: run1 = fit Train, chọn siêu tham số và dừng sớm trên Val, chấm Test; run2 = fit lại Train+Val với cấu hình đã chọn, chấm Test.

Bằng chứng cho thứ hạng yếu tố gồm: (1) kiểm định Kruskal-Wallis và kích thước hiệu ứng, (2) permutation importance theo nhóm trên Test, (3) refit bỏ nhóm và refit chỉ giữ nhóm.
Notebook chạy trên CPU.

In [ ]:
import os, sys, glob, json, random, warnings, time, shutil, subprocess
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error
for _p in ['xgboost', 'lightgbm', 'optuna']:
    try: __import__(_p)
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=True)
import xgboost as xgb, lightgbm as lgb, optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.width', 250, 'display.max_columns', None, 'display.max_rows', 300)

FAST_MODE = False
SEED = 42
TRAIN_END = pd.Timestamp('2019-10-31 23:59:59')
VAL_END   = pd.Timestamp('2020-01-15 23:59:59')      # Test = 2020-01-16 -> end of data
N_TRIALS  = {'ridge': 12, 'rf': 10, 'xgb': 20, 'lgbm': 20}
NTREE_RF  = 250
PERM_REPEATS = 20
N_BOOT = 300
if FAST_MODE:
    N_TRIALS = {'ridge': 2, 'rf': 2, 'xgb': 2, 'lgbm': 2}; NTREE_RF = 40; PERM_REPEATS = 3; N_BOOT = 30
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3, 'axes.spines.top': False, 'axes.spines.right': False})

MODEL_TAG = 'rq1'
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './outputs_rq1'
FIG_DIR = os.path.join(OUT_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, f'rq1_{name}.png'), dpi=140, bbox_inches='tight'); plt.show()
def resolve_data():
    pats = ['/kaggle/input/**/acn_data*.xlsx', '/kaggle/input/**/acn*.xlsx', './acn*.xlsx', '../acn*.xlsx',
            '../../acn*.xlsx', '/mnt/user-data/uploads/**/acn*.xlsx']
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits: return hits[0]
    raise FileNotFoundError('acn*.xlsx not found. Add the ACN dataset as notebook input.')
DATA_PATH = resolve_data()
XGB_DEVICE = 'cuda' if shutil.which('nvidia-smi') else 'cpu'
print('data:', DATA_PATH, '| xgboost device:', XGB_DEVICE, '| FAST_MODE:', FAST_MODE)

In [ ]:
# ---------- 1. Read and clean sessions (same rules as the RQ2 notebooks) ----------
try: raw = pd.read_excel(DATA_PATH, sheet_name='full_data')
except ValueError: raw = pd.read_excel(DATA_PATH)
for c in ['connectionTime', 'disconnectTime', 'doneChargingTime']: raw[c] = pd.to_datetime(raw[c])
df = raw.drop_duplicates(['stationID', 'connectionTime']).copy()
df = df[(df.kWhDelivered > 0) & (df.disconnectTime > df.connectionTime)].sort_values('connectionTime').reset_index(drop=True)
print(f'sessions: {len(df):,} (raw {len(raw):,}) | sites: {df.siteID.value_counts().to_dict()}')

# ---------- 2. Features known at arrival ----------
hol = USFederalHolidayCalendar().holidays('2018-01-01', '2021-12-31')
ct = df.connectionTime
df['hour_int']  = ct.dt.hour
hf = ct.dt.hour + ct.dt.minute / 60
df['hour_sin'] = np.sin(2 * np.pi * hf / 24); df['hour_cos'] = np.cos(2 * np.pi * hf / 24)
df['dow'] = ct.dt.dayofweek
df['is_weekend'] = (df.dow >= 5).astype(int)
df['is_holiday'] = ct.dt.normalize().isin(hol).astype(int)          # US federal calendar, not the file's column
df['month'] = ct.dt.month
df['is_workplace'] = df.siteID.isin([2, 19]).astype(int)            # site 1 = campus (public access), 2 and 19 = workplace
for s in [1, 2, 19]: df[f'site_{s}'] = (df.siteID == s).astype(int)
df['location'] = np.where(df.is_workplace == 1, 'Workplace', 'Campus (public)')
df['parking_hours'] = (df.disconnectTime - df.connectionTime).dt.total_seconds() / 3600

FEATS = ['hour_int', 'hour_sin', 'hour_cos', 'dow', 'is_weekend', 'is_holiday', 'month',
         'is_workplace', 'site_1', 'site_2', 'site_19']
GROUPS = {'Time of day': ['hour_int', 'hour_sin', 'hour_cos'],
          'Day of week': ['dow', 'is_weekend'],
          'Location type': ['is_workplace', 'site_1', 'site_2', 'site_19'],
          'Season (month)': ['month'],
          'Holiday': ['is_holiday']}
assert sorted(sum(GROUPS.values(), [])) == sorted(FEATS)

tr = df[df.connectionTime <= TRAIN_END]; va = df[(df.connectionTime > TRAIN_END) & (df.connectionTime <= VAL_END)]
te = df[df.connectionTime > VAL_END]; dev = pd.concat([tr, va])
print(pd.DataFrame({'sessions': [len(tr), len(va), len(te)], 'first': [tr.connectionTime.min(), va.connectionTime.min(), te.connectionTime.min()],
                    'last': [tr.connectionTime.max(), va.connectionTime.max(), te.connectionTime.max()]}, index=['train', 'val', 'test']))
print(df.groupby('location').kWhDelivered.agg(['count', 'mean', 'median', 'std']).round(3))

In [ ]:
# ---------- 3. Descriptive tables + classical tests ----------
DESC_ROWS = []
def add_desc(table, key, loc, g):
    DESC_ROWS.append(dict(table=table, key=key, location=loc, n=len(g), mean_kWh=g.kWhDelivered.mean(),
                          median_kWh=g.kWhDelivered.median(), std_kWh=g.kWhDelivered.std()))
for loc, g in [('All', df)] + list(df.groupby('location')):
    for h, gg in g.groupby('hour_int'): add_desc('by_arrival_hour', int(h), loc, gg)
    for d, gg in g.groupby('dow'): add_desc('by_day_of_week', int(d), loc, gg)
    for wk, gg in g.groupby('is_weekend'): add_desc('weekday_vs_weekend', 'weekend' if wk else 'weekday', loc, gg)
for s, gg in df.groupby('siteID'): add_desc('by_site', int(s), 'All', gg)
for l, gg in df.groupby('location'): add_desc('by_location_type', l, 'All', gg)
DESC_TBL = pd.DataFrame(DESC_ROWS); DESC_TBL.to_csv(os.path.join(OUT_DIR, 'rq1_descriptive_tables.csv'), index=False)

BINS = [0, 6, 9, 12, 15, 18, 24]; BIN_LAB = ['00-06', '06-09', '09-12', '12-15', '15-18', '18-24']
df['hour_bin'] = pd.cut(df.hour_int, BINS, right=False, labels=BIN_LAB)

def kruskal_eff(groups):
    groups = [g for g in groups if len(g) > 1]; n = sum(len(g) for g in groups)
    H, p = stats.kruskal(*groups); eps2 = H * (n + 1) / (n ** 2 - 1)      # epsilon-squared (Tomczak and Tomczak)
    return float(H), float(p), float(eps2), int(n), len(groups)
def mw_eff(a, b):
    U, p = stats.mannwhitneyu(a, b, alternative='two-sided'); delta = 2 * U / (len(a) * len(b)) - 1   # Cliff's delta
    return float(U), float(p), float(delta)

TEST_ROWS = []
for loc, g in [('All', df)] + list(df.groupby('location')):
    for fac, col in [('Time of day (6 bins)', 'hour_bin'), ('Day of week (7)', 'dow')]:
        H, p, e2, n, k = kruskal_eff([x.kWhDelivered.values for _, x in g.groupby(col, observed=True)])
        TEST_ROWS.append(dict(factor=fac, subset=loc, test='Kruskal-Wallis', statistic=H, p_value=p, effect_size=e2, effect_name='epsilon_sq', n=n, groups=k))
    a = g[g.is_weekend == 0].kWhDelivered.values; b = g[g.is_weekend == 1].kWhDelivered.values
    if len(a) > 1 and len(b) > 1:
        U, p, d = mw_eff(a, b); TEST_ROWS.append(dict(factor='Weekday vs weekend', subset=loc, test='Mann-Whitney U', statistic=U, p_value=p, effect_size=d, effect_name='cliffs_delta', n=len(g), groups=2))
H, p, e2, n, k = kruskal_eff([x.kWhDelivered.values for _, x in df.groupby('siteID')])
TEST_ROWS.append(dict(factor='Site (3)', subset='All', test='Kruskal-Wallis', statistic=H, p_value=p, effect_size=e2, effect_name='epsilon_sq', n=n, groups=k))
U, p, d = mw_eff(df[df.is_workplace == 1].kWhDelivered.values, df[df.is_workplace == 0].kWhDelivered.values)
TEST_ROWS.append(dict(factor='Workplace vs campus', subset='All', test='Mann-Whitney U', statistic=U, p_value=p, effect_size=d, effect_name='cliffs_delta', n=len(df), groups=2))
STATS_TBL = pd.DataFrame(TEST_ROWS); STATS_TBL.to_csv(os.path.join(OUT_DIR, 'rq1_stats_tests.csv'), index=False)
print(STATS_TBL.round(4).to_string())
print('\nMean kWh by location x weekend:'); print(df.pivot_table(index='location', columns='is_weekend', values='kWhDelivered', aggfunc=['mean', 'count']).round(2))

In [ ]:
# ---------- 4. Descriptive figures ----------
COL = {'Campus (public)': '#0072B2', 'Workplace': '#D55E00'}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(df.kWhDelivered, bins=60, color='#555', alpha=.85); ax[0].axvline(df.kWhDelivered.mean(), color='r', ls='--', lw=1, label=f'mean {df.kWhDelivered.mean():.2f}')
ax[0].set(xlabel='Energy delivered per session (kWh)', ylabel='Sessions', title='Distribution of energy per session'); ax[0].legend(fontsize=8)
for loc, g in df.groupby('location'): ax[1].hist(g.kWhDelivered, bins=50, alpha=.55, density=True, color=COL[loc], label=loc)
ax[1].set(xlabel='Energy delivered per session (kWh)', ylabel='Density', title='By location type'); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig('F1_kwh_distribution')

def mean_ci(g, col):
    m = g.groupby(col).kWhDelivered.agg(['mean', 'sem', 'count']); return m
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
for loc, g in df.groupby('location'):
    m = mean_ci(g, 'hour_int'); ax[0].plot(m.index, m['mean'], 'o-', ms=3, color=COL[loc], label=loc)
    ax[0].fill_between(m.index, m['mean'] - 1.96 * m['sem'].fillna(0), m['mean'] + 1.96 * m['sem'].fillna(0), color=COL[loc], alpha=.15)
    n = g.groupby('hour_int').size(); ax[1].plot(n.index, n.values, 'o-', ms=3, color=COL[loc], label=loc)
ax[0].set(xlabel='Arrival hour', ylabel='Mean kWh per session', title='Mean energy by arrival hour'); ax[0].legend(fontsize=8)
ax[1].set(xlabel='Arrival hour', ylabel='Sessions', title='Sessions by arrival hour'); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig('F2_by_arrival_hour')

fig, ax = plt.subplots(figsize=(6.5, 3.6)); w = 0.38; days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
for k, (loc, g) in enumerate(df.groupby('location')):
    m = mean_ci(g, 'dow').reindex(range(7)); ax.bar(np.arange(7) + (k - .5) * w, m['mean'], w, yerr=1.96 * m['sem'], color=COL[loc], label=loc, capsize=2)
ax.set_xticks(range(7)); ax.set_xticklabels(days); ax.set(ylabel='Mean kWh per session', title='Mean energy by day of week'); ax.legend(fontsize=8)
plt.tight_layout(); savefig('F3_by_day_of_week')

In [ ]:
# ---------- 5. Models: factory, tuning space, fit/predict ----------
CAT_COLS = ['hour_int', 'dow', 'month']; NUM_COLS = [c for c in FEATS if c not in CAT_COLS]
def make_ridge(alpha):
    ct_ = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS)], remainder='passthrough')
    return Pipeline([('ct', ct_), ('ridge', Ridge(alpha=alpha))])

def space(kind, t):
    if kind == 'ridge': return dict(alpha=t.suggest_float('alpha', 1e-2, 1e3, log=True))
    if kind == 'rf': return dict(max_depth=t.suggest_int('max_depth', 4, 24), min_samples_leaf=t.suggest_int('min_samples_leaf', 2, 60, log=True),
                                  max_features=t.suggest_float('max_features', 0.4, 1.0))
    if kind == 'xgb': return dict(max_depth=t.suggest_int('max_depth', 2, 10), learning_rate=t.suggest_float('learning_rate', 0.01, 0.3, log=True),
                                   subsample=t.suggest_float('subsample', 0.5, 1.0), colsample_bytree=t.suggest_float('colsample_bytree', 0.5, 1.0),
                                   min_child_weight=t.suggest_int('min_child_weight', 1, 50), reg_lambda=t.suggest_float('reg_lambda', 1e-2, 100, log=True),
                                   reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True))
    return dict(num_leaves=t.suggest_int('num_leaves', 4, 128, log=True), learning_rate=t.suggest_float('learning_rate', 0.01, 0.3, log=True),
                min_child_samples=t.suggest_int('min_child_samples', 5, 200), subsample=t.suggest_float('subsample', 0.5, 1.0),
                colsample_bytree=t.suggest_float('colsample_bytree', 0.5, 1.0), reg_lambda=t.suggest_float('reg_lambda', 1e-2, 100, log=True),
                reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True))

def fit_model(kind, p, Xtr, ytr, Xva=None, yva=None, n_iter=None):
    # returns (model, n_iter). Boosters: early stopping on (Xva, yva) when n_iter is None, else fixed n_iter.
    if kind == 'ridge': return make_ridge(p['alpha']).fit(Xtr, ytr), None
    if kind == 'rf':
        return RandomForestRegressor(n_estimators=NTREE_RF, n_jobs=-1, random_state=SEED, **p).fit(Xtr, ytr), None
    if kind == 'xgb':
        if n_iter is None:
            m = xgb.XGBRegressor(n_estimators=3000, early_stopping_rounds=50, tree_method='hist', device=XGB_DEVICE, random_state=SEED, n_jobs=-1, **p)
            m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False); return m, int(m.best_iteration) + 1
        return xgb.XGBRegressor(n_estimators=int(n_iter), tree_method='hist', device=XGB_DEVICE, random_state=SEED, n_jobs=-1, **p).fit(Xtr, ytr), int(n_iter)
    if n_iter is None:
        m = lgb.LGBMRegressor(n_estimators=3000, subsample_freq=1, random_state=SEED, verbosity=-1, n_jobs=-1, **p)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)]); return m, int(m.best_iteration_)
    return lgb.LGBMRegressor(n_estimators=int(n_iter), subsample_freq=1, random_state=SEED, verbosity=-1, n_jobs=-1, **p).fit(Xtr, ytr), int(n_iter)

def predict(m, X): return np.clip(m.predict(X), 0, None)
def metrics(y, p): return dict(MAE=mean_absolute_error(y, p), RMSE=float(np.sqrt(mean_squared_error(y, p))), R2=r2_score(y, p), MedAE=median_absolute_error(y, p))
def metrics_se(y, p, B=None, seed=SEED):
    B = B or N_BOOT; rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); pt = metrics(y, p); bs = {k: [] for k in pt}
    for _ in range(B):
        i = rng.integers(0, len(y), len(y)); mm = metrics(y[i], p[i])
        for k in mm: bs[k].append(mm[k])
    return {k: (pt[k], float(np.std(bs[k], ddof=1))) for k in pt}

Xtr, ytr = tr[FEATS], tr.kWhDelivered.values; Xva, yva = va[FEATS], va.kWhDelivered.values
Xte, yte = te[FEATS], te.kWhDelivered.values; Xdev, ydev = dev[FEATS], dev.kWhDelivered.values
KINDS = ['ridge', 'rf', 'xgb', 'lgbm']; NAMES = {'ridge': 'Ridge (linear)', 'rf': 'Random Forest', 'xgb': 'XGBoost', 'lgbm': 'LightGBM'}

# ---------- 6. Tuning on Val only (Train -> Val) ----------
BEST, NITER, TUNE_ROWS = {}, {}, []
for kind in KINDS:
    def obj(t, kind=kind):
        p = space(kind, t); m, n = fit_model(kind, p, Xtr, ytr, Xva, yva)
        t.set_user_attr('n_iter', n); return float(np.sqrt(mean_squared_error(yva, predict(m, Xva))))
    st = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.time(); st.optimize(obj, n_trials=N_TRIALS[kind])
    BEST[kind] = dict(st.best_params); NITER[kind] = st.best_trial.user_attrs.get('n_iter')
    for tr_ in st.trials: TUNE_ROWS.append(dict(model=NAMES[kind], trial=tr_.number, val_rmse=tr_.value, n_iter=tr_.user_attrs.get('n_iter'), **tr_.params))
    print(f'{NAMES[kind]:16s} best val RMSE {st.best_value:.4f} | {N_TRIALS[kind]} trials | {time.time()-t0:.0f}s | {BEST[kind]} | iters {NITER[kind]}')
pd.DataFrame(TUNE_ROWS).to_csv(os.path.join(OUT_DIR, 'rq1_tuning_trials.csv'), index=False)

In [ ]:
# ---------- 7. Two protocols ----------
# run1: fit on Train with the tuned settings (boosters stop at the iteration found on Val), evaluate Val and Test
# run2: refit on Train+Val (iterations x 1.15 because there is more data), evaluate Test
RES_ROWS, MODELS1, MODELS2, PRED_TEST = [], {}, {}, {}
for kind in KINDS:
    m1, _ = fit_model(kind, BEST[kind], Xtr, ytr, n_iter=NITER[kind]); MODELS1[kind] = m1
    n2 = None if NITER[kind] is None else max(10, int(round(NITER[kind] * 1.15)))
    m2, _ = fit_model(kind, BEST[kind], Xdev, ydev, n_iter=n2); MODELS2[kind] = m2
    for run, m in [('run1', m1), ('run2', m2)]:
        groups = [('Test', Xte, yte)] + ([('Val', Xva, yva)] if run == 'run1' else [])
        for gname, X_, y_ in groups:
            p_ = predict(m, X_); 
            if gname == 'Test': PRED_TEST[(kind, run)] = p_
            for k, (v, se) in metrics_se(y_, p_).items():
                RES_ROWS.append(dict(model=NAMES[kind], run=run, group=gname, metric=k, value=v, SE=se))
# naive reference: predict the training mean
for run, y_fit in [('run1', ytr), ('run2', ydev)]:
    p_ = np.full(len(yte), y_fit.mean())
    for k, (v, se) in metrics_se(yte, p_).items(): RES_ROWS.append(dict(model='Mean predictor', run=run, group='Test', metric=k, value=v, SE=se))
RES = pd.DataFrame(RES_ROWS); RES.to_csv(os.path.join(OUT_DIR, 'rq1_results_models.csv'), index=False)
TW = RES[RES.group == 'Test'].pivot_table(index=['model', 'run'], columns='metric', values='value')[['MAE', 'RMSE', 'R2', 'MedAE']]
print('=== Test metrics ==='); print(TW.round(4).to_string())
VW = RES[(RES.group == 'Val')].pivot_table(index='model', columns='metric', values='value')[['MAE', 'RMSE', 'R2', 'MedAE']]
print('\n=== Val metrics (run1) ==='); print(VW.round(4).to_string())
# the best model is chosen on Val (run1 R2), not on Test
BEST_KIND = max(KINDS, key=lambda k: VW.loc[NAMES[k], 'R2']); print('\nBest model on Val:', NAMES[BEST_KIND])

# sensitivity model: adds parking duration (known only at departure, so NOT a forecasting model)
SENS_FEATS = FEATS + ['parking_hours']
ms, ns = fit_model('lgbm', BEST['lgbm'], tr[SENS_FEATS], ytr, va[SENS_FEATS], yva)
msf, _ = fit_model('lgbm', BEST['lgbm'], dev[SENS_FEATS], ydev, n_iter=max(10, int(ns * 1.15)))
SENS = metrics(yte, predict(msf, te[SENS_FEATS])); print('LightGBM + parking duration (Test, run2):', {k: round(v, 4) for k, v in SENS.items()})

In [ ]:
# ---------- 8. Grouped permutation importance on Test (best model, run2) ----------
def grouped_permutation(model, X, y, groups, repeats, seed=SEED):
    rng = np.random.default_rng(seed); base = predict(model, X); b_r2, b_mae = r2_score(y, base), mean_absolute_error(y, base); rows = []
    for g, cols in groups.items():
        d_r2, d_mae = [], []
        for _ in range(repeats):
            perm = rng.permutation(len(X)); Xp = X.copy(); Xp[cols] = X[cols].values[perm]; pp = predict(model, Xp)
            d_r2.append(b_r2 - r2_score(y, pp)); d_mae.append(mean_absolute_error(y, pp) - b_mae)
        rows.append(dict(group=g, delta_R2_mean=np.mean(d_r2), delta_R2_sd=np.std(d_r2, ddof=1), delta_MAE_mean=np.mean(d_mae), delta_MAE_sd=np.std(d_mae, ddof=1)))
    return pd.DataFrame(rows), b_r2, b_mae
best_model = MODELS2[BEST_KIND]
PERM, b_r2, b_mae = grouped_permutation(best_model, Xte, yte, GROUPS, PERM_REPEATS)
PERM['share_of_total_%'] = 100 * PERM.delta_R2_mean.clip(lower=0) / PERM.delta_R2_mean.clip(lower=0).sum()
PERM['rank'] = PERM.delta_R2_mean.rank(ascending=False).astype(int); PERM.insert(0, 'model', NAMES[BEST_KIND]); PERM = PERM.sort_values('rank')
PERM.to_csv(os.path.join(OUT_DIR, 'rq1_permutation_importance.csv'), index=False)
print(f'Base Test R2 {b_r2:.4f}, MAE {b_mae:.4f}'); print(PERM.round(4).to_string(index=False))
# permutation importance for the other models, as a robustness check
PERM_ALL = []
for kind in KINDS:
    d_, _, _ = grouped_permutation(MODELS2[kind], Xte, yte, GROUPS, max(3, PERM_REPEATS // 4)); d_.insert(0, 'model', NAMES[kind]); PERM_ALL.append(d_)
PERM_ALL = pd.concat(PERM_ALL); PERM_ALL.to_csv(os.path.join(OUT_DIR, 'rq1_permutation_importance_all_models.csv'), index=False)

# ---------- 9. Refit without a group / with only one group (Train -> Val early stopping -> Test) ----------
def refit_eval(cols):
    p = BEST[BEST_KIND]; m, n = fit_model(BEST_KIND, p, tr[cols], ytr, va[cols], yva) if BEST_KIND in ('xgb', 'lgbm') else fit_model(BEST_KIND, p, tr[cols], ytr)
    return metrics(yte, predict(m, te[cols]))
DROP_ROWS = []; full = refit_eval(FEATS); DROP_ROWS.append(dict(setting='All features', R2=full['R2'], MAE=full['MAE'], dR2_vs_full=0.0))
for g, cols in GROUPS.items():
    keep = [c for c in FEATS if c not in cols]; r = refit_eval(keep)
    DROP_ROWS.append(dict(setting=f'Drop: {g}', R2=r['R2'], MAE=r['MAE'], dR2_vs_full=full['R2'] - r['R2']))
for g, cols in GROUPS.items():
    r = refit_eval(cols); DROP_ROWS.append(dict(setting=f'Only: {g}', R2=r['R2'], MAE=r['MAE'], dR2_vs_full=full['R2'] - r['R2']))
DROP = pd.DataFrame(DROP_ROWS); DROP.insert(0, 'model', NAMES[BEST_KIND]); DROP.to_csv(os.path.join(OUT_DIR, 'rq1_group_dropout.csv'), index=False)
print(DROP.round(4).to_string(index=False))

In [ ]:
# ---------- 10. Optional SHAP (tree models only, skipped if the package is unavailable) ----------
SHAP_TBL = None
try:
    try: import shap
    except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap'], check=True); import shap
    if BEST_KIND in ('xgb', 'lgbm', 'rf'):
        samp = Xte.sample(min(2000, len(Xte)), random_state=SEED); sv = shap.TreeExplainer(best_model).shap_values(samp)
        SHAP_TBL = pd.DataFrame({'feature': FEATS, 'mean_abs_shap': np.abs(sv).mean(axis=0)}).sort_values('mean_abs_shap', ascending=False)
        SHAP_TBL.to_csv(os.path.join(OUT_DIR, 'rq1_shap_mean_abs.csv'), index=False); print(SHAP_TBL.round(4).to_string(index=False))
except Exception as e:
    print('SHAP skipped:', repr(e)[:200])

In [ ]:
# ---------- 11. Result figures ----------
P = PERM.sort_values('delta_R2_mean')
fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
ax[0].barh(P.group, P.delta_R2_mean, xerr=P.delta_R2_sd, color='#0072B2', capsize=3); ax[0].set(xlabel='Drop in Test R2 when the group is shuffled', title=f'Permutation importance ({NAMES[BEST_KIND]})')
P2 = PERM.sort_values('delta_MAE_mean'); ax[1].barh(P2.group, P2.delta_MAE_mean, xerr=P2.delta_MAE_sd, color='#D55E00', capsize=3); ax[1].set(xlabel='Increase in Test MAE (kWh)', title='Same, measured in MAE')
plt.tight_layout(); savefig('F4_permutation_importance')

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8)); mods = [NAMES[k] for k in KINDS]; w = 0.38
for j, (met, ttl) in enumerate([('R2', 'Test R2'), ('MAE', 'Test MAE (kWh)')]):
    for k, run in enumerate(['run1', 'run2']):
        v = [RES[(RES.model == m) & (RES.run == run) & (RES.group == 'Test') & (RES.metric == met)].value.iloc[0] for m in mods]
        s = [RES[(RES.model == m) & (RES.run == run) & (RES.group == 'Test') & (RES.metric == met)].SE.iloc[0] for m in mods]
        ax[j].bar(np.arange(len(mods)) + (k - .5) * w, v, w, yerr=s, capsize=2, label={'run1': 'Train, tune on Val', 'run2': 'Train+Val refit'}[run])
    ax[j].set_xticks(range(len(mods))); ax[j].set_xticklabels(mods, rotation=15, fontsize=8); ax[j].set(title=ttl); ax[j].legend(fontsize=7)
plt.tight_layout(); savefig('F5_model_comparison')

p_best = PRED_TEST[(BEST_KIND, 'run2')]
fig, ax = plt.subplots(figsize=(4.6, 4.2)); ax.scatter(yte, p_best, s=3, alpha=.2, color='#009E73'); mx = max(yte.max(), p_best.max())
ax.plot([0, mx], [0, mx], 'k--', lw=.8); ax.set(xlabel='Actual kWh per session', ylabel='Predicted kWh', title=f'{NAMES[BEST_KIND]}: Test'); plt.tight_layout(); savefig('F6_actual_vs_pred')

D2 = DROP.copy(); fig, ax = plt.subplots(figsize=(6.5, 4)); ax.barh(D2.setting, D2.R2, color=['#444'] + ['#0072B2'] * 5 + ['#D55E00'] * 5)
ax.set(xlabel='Test R2 (refit)', title='Dropping or keeping a single factor group'); ax.invert_yaxis(); plt.tight_layout(); savefig('F7_group_dropout')

# ---------- 12. Summary for the paper ----------
def g(model, run, grp, met, col='value'): return float(RES[(RES.model == model) & (RES.run == run) & (RES.group == grp) & (RES.metric == met)][col].iloc[0])
SUMMARY = {'n_sessions': int(len(df)), 'n_train': int(len(tr)), 'n_val': int(len(va)), 'n_test': int(len(te)),
           'mean_kWh': float(df.kWhDelivered.mean()), 'median_kWh': float(df.kWhDelivered.median()), 'std_kWh': float(df.kWhDelivered.std()),
           'best_model_on_val': NAMES[BEST_KIND],
           'test_metrics': {NAMES[k]: {run: {m: g(NAMES[k], run, 'Test', m) for m in ['MAE', 'RMSE', 'R2', 'MedAE']} for run in ['run1', 'run2']} for k in KINDS},
           'test_metrics_se': {NAMES[k]: {run: {m: g(NAMES[k], run, 'Test', m, 'SE') for m in ['MAE', 'RMSE', 'R2']} for run in ['run1', 'run2']} for k in KINDS},
           'sensitivity_with_parking_duration_test': SENS,
           'factor_ranking_permutation': PERM[['group', 'delta_R2_mean', 'delta_R2_sd', 'delta_MAE_mean', 'share_of_total_%', 'rank']].to_dict('records'),
           'group_dropout': DROP.to_dict('records'),
           'effect_sizes': STATS_TBL[['factor', 'subset', 'test', 'effect_size', 'effect_name', 'p_value']].to_dict('records'),
           'weekend_share_%': float(100 * df.is_weekend.mean()), 'peak_arrival_hour': int(df.hour_int.value_counts().idxmax()),
           'location_counts': df.location.value_counts().to_dict()}
with open(os.path.join(OUT_DIR, 'rq1_best_hyperparameters.json'), 'w') as f:
    json.dump({'best_params': BEST, 'n_iter_run1': NITER, 'tuning': 'Optuna TPE, objective = Val RMSE (Train -> Val)', 'n_trials': N_TRIALS,
               'split': {'TRAIN_END': str(TRAIN_END), 'VAL_END': str(VAL_END)}}, f, indent=2, default=str)
with open(os.path.join(OUT_DIR, 'rq1_summary.json'), 'w') as f: json.dump(SUMMARY, f, indent=2, default=str)
print(json.dumps({k: SUMMARY[k] for k in ['best_model_on_val', 'factor_ranking_permutation']}, indent=1, default=str))
print('Saved:', sorted(os.listdir(OUT_DIR)))

## Cách đọc kết quả

1. Bảng Test cho thấy mô hình nào dự đoán kWh mỗi phiên tốt nhất; mô hình tốt nhất được chọn bằng R2 trên Val, không dùng Test để chọn.
2. Permutation importance theo nhóm cho biết nhóm yếu tố nào làm giảm R2 nhiều nhất khi bị xáo trộn. Giá trị lớn hơn nghĩa là mô hình phụ thuộc nhiều hơn vào nhóm đó.
3. Bảng refit bỏ nhóm và chỉ giữ nhóm là bằng chứng độc lập với permutation, vì mô hình được huấn luyện lại.
4. Kiểm định Kruskal-Wallis và epsilon-squared chỉ mô tả sự khác biệt của kWh theo từng yếu tố riêng lẻ, không loại bỏ nhiễu chéo giữa các yếu tố (ví dụ loại địa điểm và ngày cuối tuần).
5. Mô hình có thêm thời gian đỗ chỉ để tham khảo: thời gian đỗ chỉ biết khi xe rời đi nên không dùng làm biến dự báo.

Giới hạn: loại địa điểm được suy ra từ site (site 1 là khuôn viên Caltech, site 2 và 19 là nơi làm việc), nên tác động của địa điểm lẫn với đặc điểm người dùng của từng site.